# Case-01:單自由度(SDOF)驗證

依 [ROADMAP.md](../ROADMAP.md) 的漸進式驗證規劃,這是本 repo 第一個實際的
OpenSeesPy 模型——刻意選最簡單的單自由度系統,確保整套建模→分析→驗證的
基本工具鏈本身沒問題,再往上疊加複雜度(Case-02 一跨一層 → ... → Case-04
桃園案例)。

**驗證方式**:同一個問題算兩次獨立於彼此的答案——閉合解手算 vs. OpenSeesPy
數值模型——只有兩者一致,才代表 OpenSeesPy 這邊的建模語法、單位、邊界條件
都設對了。這不是在算一個新結構,是在測試「测量工具」本身準不準。

## 第 1 課:問題定義與手算閉合解

單自由度系統:質量 m,水平勁度 K,底部固定。

自然頻率與週期:

$$\omega = \sqrt{K/m}, \qquad T = \frac{2\pi}{\omega}$$

靜力關係(虎克定律):

$$x = F/K$$

這兩條公式本身沒有近似,是精確解——後面 OpenSeesPy 算出的結果理論上要
**完全吻合**,不是「差不多」,任何非零誤差都代表建模有問題。

In [1]:
import math
import openseespy.opensees as ops

# ---- 系統參數 ----
m_kg   = 50_000.0       # 質量, 50公噸 = 50,000 kg
K_Npm  = 5_000_000.0    # 水平勁度, 5000 kN/m = 5,000,000 N/m

# ---- 手算閉合解 ----
omega_hand = math.sqrt(K_Npm / m_kg)
T_hand = 2*math.pi/omega_hand

print(f"m = {m_kg/1000:.1f} 公噸,  K = {K_Npm/1000:.1f} kN/m")
print(f"手算: omega = {omega_hand:.6f} rad/s,  T = {T_hand:.6f} s")

m = 50.0 公噸,  K = 5000.0 kN/m
手算: omega = 10.000000 rad/s,  T = 0.628319 s


## 第 2 課:OpenSeesPy 建模

用 `zeroLength` 元素模擬一個彈簧——兩個座標重合的節點,一個固定(基礎),
一個只留水平自由度並掛質量(代表 SDOF 的集中質量)。

這是 OpenSeesPy 裡最基本的元素類型,刻意不用真實的柱元素(`elasticBeamColumn`),
避免斷面性質、勁度換算這些額外變數混進第一個驗證案例裡——每一個 Case 只
新增一個新的複雜度,這裡要驗證的只有「彈簧-質量系統本身」。

In [2]:
ops.wipe()
ops.model('basic', '-ndm', 2, '-ndf', 3)

ops.node(1, 0.0, 0.0)   # 基礎節點
ops.node(2, 0.0, 0.0)   # 質量節點(座標重合,用zeroLength彈簧連接)

ops.fix(1, 1, 1, 1)     # 基礎完全固定
ops.fix(2, 0, 1, 1)     # 質量節點只留水平自由度(Ux)

ops.mass(2, m_kg, 0.0, 0.0)

ops.uniaxialMaterial('Elastic', 1, K_Npm)
ops.element('zeroLength', 1, 1, 2, '-mat', 1, '-dir', 1)

print("模型建立完成: 2節點, 1個zeroLength彈簧元素, 質量掛在節點2")

模型建立完成: 2節點, 1個zeroLength彈簧元素, 質量掛在節點2


## 第 3 課:特徵值分析驗證自然週期

用 OpenSeesPy 的特徵值分析求出這個模型「自己算出來」的自然頻率,
跟第 1 課的手算閉合解對比。

> 技術備註:單自由度系統只有 1 個自由度,OpenSeesPy 預設的 ARPACK
> 特徵值求解器(`eigen(1)`)對這麼小的系統會直接報錯(`NCV must be
> greater than NEV`,這是 ARPACK 演算法對系統規模的內建限制),
> 這裡改用 `-fullGenLapack` 選項——對大型系統會慢很多,但對這種
> 教學/驗證用的小系統完全沒有影響。

In [3]:
eigVals = ops.eigen('-fullGenLapack', 1)
omega_ops = math.sqrt(eigVals[0])
T_ops = 2*math.pi/omega_ops

print(f"OpenSeesPy 特徵值分析: omega = {omega_ops:.6f} rad/s,  T = {T_ops:.6f} s")

rel_err_T = abs(T_ops - T_hand) / T_hand
print(f"與手算相對誤差 = {rel_err_T:.2e}")

assert rel_err_T < 1e-9, "週期不吻合,建模有問題!"
print("[PASS] 自然週期與手算閉合解完全吻合")

OpenSeesPy 特徵值分析: omega = 10.000000 rad/s,  T = 0.628319 s
與手算相對誤差 = 0.00e+00
[PASS] 自然週期與手算閉合解完全吻合


WARNING - the 'fullGenLapack' eigen solver is VERY SLOW. Consider using the default eigen solver.

## 第 4 課:靜力測試驗證勁度定義

再用另一個獨立的檢查角度:直接施加水平力,看模型算出的位移是否符合
$x=F/K$。這跟第 3 課的特�徵值分析是兩個不同的驗證路徑——一個測動態
行為(週期),一個測靜態行為(勁度定義本身對不對),用意是避免「週期
剛好吻合但其實勁度設錯,只是誤打誤撞」這種情況。

In [4]:
F_N = 100_000.0  # 施加水平力, 100 kN

ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
ops.load(2, F_N, 0.0, 0.0)

ops.system('BandGeneral')
ops.numberer('RCM')
ops.constraints('Plain')
ops.test('NormDispIncr', 1e-10, 20)
ops.algorithm('Newton')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')
ops.analyze(1)

ux_ops = ops.nodeDisp(2, 1)
ux_hand = F_N / K_Npm

print(f"施加水平力 F = {F_N/1000:.1f} kN")
print(f"OpenSeesPy 位移 = {ux_ops:.8f} m")
print(f"手算 F/K 位移   = {ux_hand:.8f} m")

rel_err_x = abs(ux_ops - ux_hand) / ux_hand
print(f"相對誤差 = {rel_err_x:.2e}")

assert rel_err_x < 1e-9, "位移不吻合,建模有問題!"
print("[PASS] 靜力位移與手算閉合解完全吻合")

施加水平力 F = 100.0 kN
OpenSeesPy 位移 = 0.02000000 m
手算 F/K 位移   = 0.02000000 m
相對誤差 = 0.00e+00
[PASS] 靜力位移與手算閉合解完全吻合


## 總結表

In [5]:
print("="*50)
print("Case-01 SDOF 驗證結果總結")
print("="*50)
print(f"{'手算週期 T':<20}{T_hand:.6f} s")
print(f"{'OpenSeesPy週期 T':<20}{T_ops:.6f} s")
print(f"{'週期相對誤差':<20}{rel_err_T:.2e}")
print(f"{'手算位移 x':<20}{ux_hand:.8f} m")
print(f"{'OpenSeesPy位移 x':<20}{ux_ops:.8f} m")
print(f"{'位移相對誤差':<20}{rel_err_x:.2e}")
print()
print("Case-01 [PASS] -- 可以進到 Case-02(一跨一層)")

Case-01 SDOF 驗證結果總結
手算週期 T              0.628319 s
OpenSeesPy週期 T      0.628319 s
週期相對誤差              0.00e+00
手算位移 x              0.02000000 m
OpenSeesPy位移 x      0.02000000 m
位移相對誤差              0.00e+00

Case-01 [PASS] -- 可以進到 Case-02(一跨一層)
